# CUDA Backend — Credit Card Pattern Matching

Compiles `cuda/card_matcher.cu` with `nvcc` and serves it over HTTPS so the
React front end can send card numbers to the GPU.

**The matching algorithm lives entirely in CUDA C.** This notebook only builds
it and puts an HTTP door in front of it.

---

### Before you run anything

**Runtime → Change runtime type → Hardware accelerator → T4 GPU**, then Save.

Without a GPU runtime the build succeeds but `cm_init()` fails at startup with
*"no CUDA device found"*.

Then: **Runtime → Run all**.

## 1. Confirm the GPU and the CUDA toolkit

In [ ]:
!nvidia-smi
print()
!nvcc --version

## 2. Fetch the source

This pulls the `cuda/` and `server/` folders from GitHub, so **push your
latest commit before running this**.

If you would rather not push, upload `card_patterns.cuh`, `card_matcher.cu`,
`selftest.cu`, `Makefile` and `app.py` by hand with the file browser on the
left and skip to step 3.

In [ ]:
import os

REPO = "https://github.com/Sayedo5/Credit-Card-Pattern-Matching-PDC-Project-.git"
ROOT = "/content/Credit-Card-Pattern-Matching-PDC-Project-"

if os.path.exists(ROOT):
    !cd {ROOT} && git pull -q
else:
    !git clone -q {REPO} {ROOT}

os.chdir(ROOT)
print("working in:", os.getcwd())
print()
!ls -1 cuda server

## 3. Compile the CUDA matcher

Builds two things:

* `libcardmatcher.so` — the shared library the Python server loads with ctypes
* `card_matcher_selftest` — a standalone GPU test binary

The Makefile emits SASS for sm_75 (Colab's T4), sm_80 and sm_86, plus PTX for
forward compatibility, so the same `.so` runs on whatever GPU Colab hands you.

In [ ]:
!cd {ROOT}/cuda && make clean && make
print()
!ls -lh {ROOT}/cuda/libcardmatcher.so

## 4. Run the GPU self-test

Every case here goes through the real kernel, not a host-side copy of the
logic. It covers one hit per rule branch in the reference table, the boundary
rejections either side of each numeric range, the per-brand length rules, and
a 1,000,000-card throughput run that cross-checks GPU output against the CPU
path card by card.

In [ ]:
!cd {ROOT}/cuda && make test

## 5. Install the server dependencies

In [ ]:
!pip install -q fastapi "uvicorn[standard]"
print("done")

## 6. Start the API and open a public tunnel

Colab cannot be reached from your laptop directly, so `cloudflared` opens a
temporary public HTTPS URL that forwards to port 8000 inside this VM. No
account or signup needed.

**The URL changes every time you run this cell.** Copy it into the React app —
click *change* at the bottom of the panel and paste it there.

In [ ]:
import json
import os
import re
import subprocess
import time
import urllib.request

ROOT = "/content/Credit-Card-Pattern-Matching-PDC-Project-"

# Clear out anything left running from a previous execution of this cell.
!pkill -f "uvicorn app:app" 2>/dev/null
!pkill -f "cloudflared tunnel" 2>/dev/null
time.sleep(1)

# ---- start the API ----------------------------------------------------
server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=f"{ROOT}/server",
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)

health = None
for _ in range(45):
    try:
        with urllib.request.urlopen("http://localhost:8000/api/health", timeout=2) as r:
            health = json.load(r)
            break
    except Exception:
        time.sleep(1)

if health is None:
    print("--- server.log ---")
    print(open("/content/server.log").read())
    raise SystemExit("the API did not come up — check the log above")

print("GPU backend ready")
print("  engine :", health["engine"])
print("  device :", health["device"])
print("  brands :", health["brands"], "networks loaded from the CUDA table")

# ---- open the tunnel --------------------------------------------------
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=open("/content/tunnel.log", "w"),
    stderr=subprocess.STDOUT,
)

PUBLIC_URL = None
for _ in range(60):
    time.sleep(1)
    match = re.search(r"https://[-\w.]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if match:
        PUBLIC_URL = match.group(0)
        break

if PUBLIC_URL is None:
    print("--- tunnel.log ---")
    print(open("/content/tunnel.log").read())
    raise SystemExit("the tunnel did not open")

print()
print("=" * 68)
print("  PASTE THIS INTO THE REACT APP  ->  click 'change' at the bottom")
print()
print("   ", PUBLIC_URL)
print()
print("=" * 68)

## 7. Verify the API through the tunnel

Each of these is a real round trip: HTTP → FastAPI → ctypes → `cudaMemcpy` →
kernel → back again.

In [ ]:
def post(path, payload):
    request = urllib.request.Request(
        PUBLIC_URL + path,
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)


probes = [
    "4111111111111111",   # Visa, 16
    "4222222222222",      # Visa, 13
    "378282246310005",    # Amex, 15
    "30569309025904",     # Diners Club, 14
    "3566002020360505",   # JCB
    "6011111111111117",   # Discover
    "6200000000000005",   # UnionPay
    "6759649826438453",   # Maestro
    "34111111111111",     # Amex prefix, wrong length
    "1234567890123456",   # no brand
]

print(f"{'number':<20} {'result':<10} {'brand':<18} {'luhn':<6} kernel")
print("-" * 68)
for number in probes:
    r = post("/api/match", {"number": number})
    print(
        f"{number:<20} "
        f"{('MATCH' if r['matched'] else 'no match'):<10} "
        f"{(r['brand']['name'] if r['brand'] else '-'):<18} "
        f"{str(r['luhnValid']):<6} "
        f"{r['kernelMs']} ms"
    )

## 8. Speedup measurement

Serial CPU versus the CUDA kernel over the same one million cards, running the
identical `cm_match()` / `cm_luhn()` code from `card_patterns.cuh`. Both
results are compared card by card, so a number here also means the two paths
agreed on every single one.

Two speedup figures are reported because they answer different questions:
`kernelSpeedup` is the compute win, `endToEndSpeedup` includes the PCIe
transfers you actually pay for.

In [ ]:
bench = post("/api/benchmark", {"cards": 1_000_000})

print("device            :", bench["device"])
print("cards             :", f"{bench['cards']:,}")
print()
print("CPU (1 thread)    :", f"{bench['cpuMs']:>9.2f} ms")
print("GPU kernel only   :", f"{bench['gpuKernelMs']:>9.2f} ms", f"  ({bench['kernelSpeedup']}x)")
print("GPU + transfers   :", f"{bench['gpuTotalMs']:>9.2f} ms", f"  ({bench['endToEndSpeedup']}x)")
print()
print("matched           :", f"{bench['matched']:,} / {bench['cards']:,}")

## Keeping it alive

The tunnel and the API stay up as long as this Colab session does. Colab
disconnects an idle notebook after roughly 90 minutes, and every disconnect
gives you a **new tunnel URL** — rerun step 6 and paste the new one into the
app.

To stop everything:

```python
!pkill -f "uvicorn app:app"; !pkill -f "cloudflared tunnel"
```